In [ ]:
from datasets import load_dataset, get_dataset_config_names
import pandas as pd

# Replace with any model from the leaderboard, e.g.:
MODEL = "mistralai__Mistral-7B-v0.1"  # format: org__modelname

DATASET = f"open-llm-leaderboard-old/details_{MODEL}"

# List all available configs (one per MMLU subject + other benchmarks)
configs = get_dataset_config_names(DATASET)
mmlu_configs = [c for c in configs if "mmlu" in c.lower()]
print(f"Found {len(mmlu_configs)} MMLU configs")

# Load all MMLU subjects and concatenate
dfs = []
for config in mmlu_configs:
    ds = load_dataset(DATASET, name=config, split="latest", trust_remote_code=True)
    df = ds.to_pandas()
    df["subject"] = config.split("mmlu_")[-1]  # extract subject name
    dfs.append(df)

mmlu_df = pd.concat(dfs, ignore_index=True)

# Each row is one question. Key columns:
# - "full_prompt": the prompt sent to the model
# - "predictions": model's predicted answer index
# - "gold_index": correct answer index
# - "acc": 1.0 if correct, 0.0 if wrong
print(mmlu_df[["subject", "gold_index", "predictions", "acc"]].head(20))
print(f"\nOverall MMLU accuracy: {mmlu_df['acc'].mean():.4f}")
print(
    f"\nPer-subject accuracy:\n{mmlu_df.groupby('subject')['acc'].mean().sort_values()}"
)